In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Không có GPU\!")

torch.cuda.empty_cache()

!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece


<>:4: SyntaxWarning: invalid escape sequence '\!'
<>:4: SyntaxWarning: invalid escape sequence '\!'
/tmp/ipykernel_22/652962394.py:4: SyntaxWarning: invalid escape sequence '\!'
  raise RuntimeError("Không có GPU\!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.


In [2]:
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "feature/change-phobert"
PROJECT_DIR = "/kaggle/working/absa-project"

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)

for d in ["data", "outputs/models", "outputs/results", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, "code/data_processing")
sys.path.insert(0, "code/phobert")


Cloning into '/kaggle/working/absa-project'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 135 (delta 18), reused 68 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 5.77 MiB | 19.95 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [3]:
import pandas as pd, os

if not os.path.exists("data/train.csv"):
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/


In [4]:
import pandas as pd, os

FORCE_REPROCESS = False

if not ((not FORCE_REPROCESS) and os.path.exists("data/train_preprocessed.csv")):
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
    segmenter.close()


In [5]:
import json, os
from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
cw = json.load(open('outputs/eda/class_weights.json'))


In [6]:
import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics = main(encoder_option='concat_4_layers', use_amp=True)


[Device] GPU: Tesla T4

PHASE PHOBERT — Multi-task ABSA
  encoder:    concat_4_layers
  seq_len:    256
  batch:      8 × 2 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...


  Dev — Epoch 18
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.2353    0.3333    0.1818    0.1481    11
HOTEL#MISCELLANEOUS                  0.3027    0.7000    0.1931    0.1714    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.4381    0.6389    0.3333    0.3194    69
* FACILITIES#MISCELLANEOUS           0.4444    1.0000    0.2857    0.4286    7
FACILITIES#GENERAL                   0.5672    0.5278    0.6129    0.2482    62
FOOD&DRINKS#PRICES                   0.6000    0.7143    0.5172    0.4055    29
ROOM_AMENITIES


  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.2353    0.3333    0.1818    0.1481    11
* FACILITIES#MISCELLANEOUS           0.2500    1.0000    0.1429    0.0952    7
HOTEL#MISCELLANEOUS                  0.3118    0.7073    0.2000    0.1757    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.4381    0.6389    0.3333    0.3194    69
FACILITIES#GENERAL                   0.5649    0.5362    0.5968    0.2482    62
FOOD&DRINKS#PRICES                   0.5882    0.6818    0.5172    0.4055    29
ROOM_AMENITIES


  Dev — Epoch 20
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.2353    0.3333    0.1818    0.1481    11
* FACILITIES#MISCELLANEOUS           0.2500    1.0000    0.1429    0.0952    7
HOTEL#MISCELLANEOUS                  0.3118    0.7073    0.2000    0.1757    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.4381    0.6389    0.3333    0.3194    69
ROOM_AMENITIES#QUALITY               0.5839    0.4943    0.7131    0.5438    122
FACILITIES#GENERAL                   0.5865    0.5493    0.6290    0.2526    62
FOOD&DRINKS#P


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
HOTEL#MISCELLANEOUS                  0.3298    0.7209    0.2138    0.1839    145
* FOOD&DRINKS#MISCELLANEOUS          0.3333    0.4286    0.2727    0.2000    11
* FACILITIES#MISCELLANEOUS           0.4444    1.0000    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4821    0.6279    0.3913    0.3617    69
ROOMS#QUALITY                        0.5000    0.3571    0.8333    0.6000    6
FOOD&DRINKS#PRICES                   0.5660    0.6250    0.5172    0.4055    29
FACILITIES#GENERAL                   0.5694    0.5000    0.6613    0.2612    62
ROOM_AMENITIES#QU


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.2410    0.6667    0.1471    0.1212    68
FACILITIES#COMFORT                   0.4390    0.6000    0.3462    0.1905    26
FACILITIES#CLEANLINESS               0.4444    0.5000    0.4000    0.2667    5
* FOOD&DRINKS#MISCELLANEOUS          0.5000    1.0000    0.3333    0.3333    3
HOTEL#QUALITY                        0.5517    0.5000    0.6154    0.4667    13
FACILITIES#GENERAL                   0.5965    0.4722    0.8095    0.9608    21
ROOMS#QUALITY     

In [7]:
import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_cls = main(encoder_option="cls_only", use_amp=True)

gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']


[Device] GPU: Tesla T4

PHASE PHOBERT — Multi-task ABSA
  encoder:    cls_only
  seq_len:    256
  batch:      8 × 2 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Tokenizer] Loaded 

[Data] Creating DataLoaders...
[DataLoader] train: 3000 samples, 375 batches, rare_oversampling=ON (alpha=2.0, power=1.0)
[DataLoader] dev: 2000 samples, 250 batches
[DataLoader] test: 600 samples, 75 batches
[Weights] 34 aspects loaded, 77 weight values clipped at 10.0

[Model] Building ABSAPhoBERT (cls_only)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[Model] 155,364,138 trainable parameters
[Scheduler] Total=3760 optimizer steps, Warmup=564
[LR] AdamW encoder_lr=1.00e-04, head_lr=5.00e-04, wd=1.00e-02, llrd=0.920, scheduler=cosine_warmup
[AMP] Mixed precision: ON
[EMA] OFF (decay=0.9990)
[Config] encoder=cls_only, seq_len=256, batch=8×2=16 (effective)

────────────────────────────────────────────────────────────
Epoch 1/20


...


  Dev — Epoch 18
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3299    0.6531    0.2207    0.1875    145
ROOMS#QUALITY                        0.3333    0.2222    0.6667    0.5333    6
* FACILITIES#MISCELLANEOUS           0.3636    0.5000    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4567    0.5000    0.4203    0.3807    69
FOOD&DRINKS#PRICES                   0.5283    0.5833    0.4828    0.3848    29
FACILITIES#GENERAL                   0.5532    0.4937    0.6290    0.2680    62
ROOM_AMENITIES


  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3141    0.6522    0.2069    0.1791    145
ROOMS#QUALITY                        0.3636    0.2500    0.6667    0.5333    6
* FACILITIES#MISCELLANEOUS           0.4000    0.6667    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4538    0.5400    0.3913    0.3617    69
FACILITIES#GENERAL                   0.5401    0.4933    0.5968    0.2596    62
FOOD&DRINKS#PRICES                   0.5490    0.6364    0.4828    0.3848    29
ROOM_AMENITIES


  Dev — Epoch 20
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3141    0.6522    0.2069    0.1791    145
ROOMS#QUALITY                        0.3636    0.2500    0.6667    0.5333    6
* FACILITIES#MISCELLANEOUS           0.4000    0.6667    0.2857    0.4286    7
FACILITIES#COMFORT                   0.4576    0.5510    0.3913    0.3617    69
FOOD&DRINKS#PRICES                   0.5490    0.6364    0.4828    0.3848    29
FACILITIES#GENERAL                   0.5507    0.5000    0.6129    0.2639    62
ROOM_AMENITIES


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FOOD&DRINKS#MISCELLANEOUS          0.1538    0.5000    0.0909    0.0833    11
HOTEL#MISCELLANEOUS                  0.3886    0.6212    0.2828    0.2360    145
ROOMS#QUALITY                        0.4211    0.3077    0.6667    0.5333    6
FACILITIES#COMFORT                   0.5000    0.4800    0.5217    0.4285    69
* FACILITIES#MISCELLANEOUS           0.5455    0.7500    0.4286    0.5000    7
FOOD&DRINKS#PRICES                   0.5517    0.5517    0.5517    0.4279    29
ROOM_AMENITIES#QUALITY               0.5569    0.4387    0.7623    0.5656    122
FACILITIES#GENER


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.3226    0.6000    0.2206    0.1769    68
* FOOD&DRINKS#MISCELLANEOUS          0.5000    1.0000    0.3333    0.3333    3
HOTEL#QUALITY                        0.5000    0.5455    0.4615    0.3704    13
FACILITIES#CLEANLINESS               0.5455    0.5000    0.6000    0.3333    5
FACILITIES#GENERAL                   0.5926    0.4848    0.7619    0.9495    21
ROOM_AMENITIES#QUALITY               0.6043    0.5250    0.7119    0.4927    59
FACILITIES#PRICES 

In [8]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open("outputs/results/training_history.json"))
best_ep = history["best_epoch"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PhoBERT concat_4_layers — Learning Curve (ABSA VLSP 2018)", fontsize=13)

ax1.plot(epochs, history["train_loss"], "o-", c="crimson",   lw=2, label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", c="steelblue", lw=2, label="Dev Loss")
ax1.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax1.set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history["dev_acd_f1"],      "s-", c="darkorange", lw=2, label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "^-", c="purple",     lw=2, label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", c="green",      lw=2.5, label="Dev Combined F1")
ax2.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax2.axhline(0.7732,  c="red",   ls=":",  alpha=0.5, label="SOTA Combined 0.7732")
ax2.set(title="F1 Score", xlabel="Epoch", ylabel="Macro F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()


In [9]:
import os, json, shutil

if os.path.exists("outputs/results/phobert_test_metrics.json"):
    m = json.load(open("outputs/results/phobert_test_metrics.json"))

if os.path.exists("outputs/results_cls_only/phobert_test_metrics.json"):
    m2 = json.load(open("outputs/results_cls_only/phobert_test_metrics.json"))

shutil.make_archive("/kaggle/working/phobert_results", "zip", "outputs")


'/kaggle/working/phobert_results.zip'